In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [9]:
spark = SparkSession.builder \
    .appName("credit-risk-lakehouse-pagamento") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

In [10]:
BASE_PATH = "/app/data"

In [11]:
df_parcela = spark.read.parquet(f"{BASE_PATH}/bronze/parcela")

df_pagamento = df_parcela \
    .filter(col("status_parcela") == "paga") \
    .select(
        monotonically_increasing_id().alias("pagamento_id"),
        col("parcela_id"),
        date_add(
            col("data_vencimento"),
            floor(rand() * 10 - 5).cast("int")
        ).alias("data_pagamento"),
        col("valor_parcela").alias("valor_pago"),
        element_at(
            array(
                lit("pix"),
                lit("boleto"),
                lit("debito_automatico"),
                lit("cartao")
            ),
            floor(rand() * 4 + 1).cast("int")
        ).alias("forma_pagamento")
    )

In [12]:
df_pagamento.show(10, truncate=False)
df_pagamento.printSchema()
df_pagamento.count()

+------------+----------+--------------+----------+-----------------+
|pagamento_id|parcela_id|data_pagamento|valor_pago|forma_pagamento  |
+------------+----------+--------------+----------+-----------------+
|0           |0         |2025-01-02    |1996.03   |cartao           |
|1           |1         |2025-01-30    |1996.03   |pix              |
|2           |2         |2025-02-25    |1996.03   |pix              |
|3           |3         |2025-03-30    |1996.03   |boleto           |
|4           |5         |2025-05-27    |1996.03   |pix              |
|5           |6         |2025-06-26    |1996.03   |pix              |
|6           |8         |2025-08-28    |1996.03   |debito_automatico|
|7           |9         |2025-09-27    |1996.03   |pix              |
|8           |10        |2025-10-26    |1996.03   |cartao           |
|9           |12        |2025-12-25    |1996.03   |debito_automatico|
+------------+----------+--------------+----------+-----------------+
only showing top 10 

225758

In [13]:
df_pagamento.select("parcela_id").distinct().count() == df_pagamento.count()

True

In [14]:
df_pagamento.write \
    .mode("overwrite") \
    .parquet("/app/data/bronze/pagamento")

In [16]:
spark.stop()